# 31. GPU Virtualization and MIG | GPU 虚拟化与 MIG

**难度：** Medium | **环境：** CPU-first | **标签：** `硬件系统`, `GPU 虚拟化`, `MIG` | **目标人群：** 硬件约束学习者

![GPU 共享、MIG 切分与服务证据](../docs/public/01_Hardware_Math_and_Systems/31_gpu_isolation_mig.svg)

---

## 本节导读

一张 GPU 同时服务多个任务时，共享会提高总体利用率，也可能放大资源争用和尾延迟。MIG 通过固定资源切片增强隔离，但切片会引入容量碎片并限制资源弹性。

本节建立“资源切片 → 容量与带宽隔离 → 吞吐与尾延迟 → 服务准入”的关系。完成后，你应能按服务目标比较共享和隔离方案，并识别容量碎片与资源争用风险。

**关键词：** `MIG`、`MPS`、多租户

---

## 前置阅读

**导语：** 这一页先把资源隔离、动态 shape 和多租户调度的关系接上，再看为什么 GPU 虚拟化必须先切边界。

- [07. CPU and GPU Heterogeneous Scheduling | CPU 与 GPU 异构调度](./07_CPU_GPU_Heterogeneous_Scheduling.ipynb)
- [29. CUDA Stream Advanced Scheduling | CUDA Stream 高级调度](./29_CUDA_Stream_Advanced_Scheduling.ipynb)
- [30. Dynamic Shape Handling | 动态 Shape 处理](./30_Dynamic_Shape_Handling.ipynb)


## Q1：GPU 虚拟化为什么首先是资源隔离问题？


多租户场景里，GPU 不只是“给谁用”，而是“怎么让很多任务共用同一块卡，同时还不互相干扰”。

如果没有隔离，某个任务可能会抢占过多算力、缓存、显存或调度窗口，导致别的任务延迟飙升、尾延迟变差。

所以 GPU 虚拟化的第一目标，不是简单提高利用率，而是先把共享资源的干扰边界定义清楚，让性能更可控。


### Q1 资源账本：每个切片实际获得多少显存与计算单元

先把“谁会干扰谁”看清楚，再谈共享效率。

In [ ]:
def partition_resources(total_memory_gb, total_compute_units, slices, reserved_memory_gb=0.0):
    """计算等分切片后的每片资源与整数计算单元碎片。"""
    if total_memory_gb <= 0 or total_compute_units <= 0 or slices <= 0:
        raise ValueError('resource totals and slices must be positive')
    if not 0 <= reserved_memory_gb < total_memory_gb:
        raise ValueError('reserved memory must be in [0, total_memory)')
    usable_memory_gb = total_memory_gb - reserved_memory_gb
    compute_units_per_slice = total_compute_units // slices
    return {
        'memory_gb_per_slice': usable_memory_gb / slices,
        'compute_units_per_slice': compute_units_per_slice,
        'unused_compute_units': total_compute_units - compute_units_per_slice * slices,
    }

for slices in [2, 3, 7]:
    print(slices, 'slices ->', partition_resources(80, 108, slices, reserved_memory_gb=2))
assert partition_resources(80, 108, 7, 2)['unused_compute_units'] == 3


## Q2：MIG 的本质是在切什么边界？


MIG 的核心，不是“把一块 GPU 拆成几块”这么简单，而是把算力、显存和部分调度边界切开，让不同租户在更稳定的资源切片里运行。

这样做的直接收益，是减少缓存和算力争用带来的抖动；间接收益，是让推理服务更容易做容量规划和 QoS 控制。


### Q2 对照验证：隔离前后的吞吐和尾延迟如何变化

资源边界越清晰，推理尾延迟通常越稳定。

In [ ]:
def isolation_delta(shared, isolated):
    """比较共享与隔离方案的吞吐、P99 和尾延迟放大。"""
    for record in (shared, isolated):
        if record['throughput_rps'] <= 0 or record['p50_ms'] <= 0 or record['p99_ms'] <= 0:
            raise ValueError('metrics must be positive')
    return {
        'throughput_change_ratio': isolated['throughput_rps'] / shared['throughput_rps'] - 1,
        'p99_reduction_ratio': 1 - isolated['p99_ms'] / shared['p99_ms'],
        'shared_tail_amplification': shared['p99_ms'] / shared['p50_ms'],
        'isolated_tail_amplification': isolated['p99_ms'] / isolated['p50_ms'],
    }

shared = {'throughput_rps': 120, 'p50_ms': 30, 'p99_ms': 150}
isolated = {'throughput_rps': 108, 'p50_ms': 32, 'p99_ms': 80}
delta = isolation_delta(shared, isolated)
assert delta['p99_reduction_ratio'] > 0
assert delta['throughput_change_ratio'] < 0
print(delta)


## Q3：为什么隔离更强，吞吐未必更高？


隔离做得越强，任务之间越不容易互相干扰，但共享的效率通常也会下降；共享做得越激进，利用率可能更高，但延迟和抖动也更容易恶化。

所以虚拟化问题的核心，不是追求“切得更细”本身，而是在隔离、吞吐和尾延迟之间找到能接受的折中。


### Q3 SLO 门槛：哪些隔离方案可以进入候选集

高利用率不一定意味着好的服务体验。

In [ ]:
def select_isolation_plan(plans, max_p99_ms, min_throughput_rps, max_jitter_ms):
    """先应用延迟、吞吐与抖动硬门槛，再选吞吐最高的方案。"""
    eligible = {}
    rejected = {}
    for name, metrics in plans.items():
        reasons = []
        if metrics['p99_ms'] > max_p99_ms:
            reasons.append('p99_slo')
        if metrics['throughput_rps'] < min_throughput_rps:
            reasons.append('throughput')
        if metrics['jitter_ms'] > max_jitter_ms:
            reasons.append('jitter')
        (rejected if reasons else eligible)[name] = reasons if reasons else metrics
    selected = max(eligible, key=lambda name: eligible[name]['throughput_rps']) if eligible else None
    return {'selected': selected, 'eligible': list(eligible), 'rejected': rejected}

plans = {
    'shared': {'throughput_rps': 120, 'p99_ms': 150, 'jitter_ms': 35},
    'mig_2way': {'throughput_rps': 108, 'p99_ms': 80, 'jitter_ms': 12},
    'mig_7way': {'throughput_rps': 70, 'p99_ms': 55, 'jitter_ms': 8},
}
decision = select_isolation_plan(plans, max_p99_ms=100, min_throughput_rps=90, max_jitter_ms=20)
assert decision['selected'] == 'mig_2way'
assert set(decision['rejected']) == {'shared', 'mig_7way'}
print(decision)


## ⚠️ 常见误区

- 虚拟化不是把 GPU 拆碎就结束了。
- MIG 关注的是隔离和可预测性，不只是拆分数量。
- 多租户推理里，延迟抖动常常比平均吞吐更重要。
- 共享和隔离本身就是一组折中。

## 相关阅读

**导语：** 如果想继续把隔离、编译和成本判断串起来，可以接着看这些页。

- [19. Operator Fusion Introduction | 算子融合导论](./19_Operator_Fusion_Introduction.ipynb)
- [32. TVM / MLIR Deep Practice | TVM / MLIR 深度实践](./32_TVM_MLIR_Deep_Practice.ipynb)
- [33. TCO and Cost Model | 算力评估与 TCO 模型](./33_TCO_and_Cost_Model.ipynb)
---